In [ ]:
from astropy.io import ascii
from astropy import units as u
from astropy.coordinates import SkyCoord # High-level coordinates
from astropy.table import Table, Column
import numpy as np
import astropy.units as u

from galpy.potential import MWPotential2014
from galpy.orbit import Orbit
from galpy.actionAngle import actionAngleStaeckel

from gala.potential import MiyamotoNagaiPotential, HernquistPotential, NFWPotential, CompositePotential
from gala.potential import MilkyWayPotential, MilkyWayPotential2022
from gala.dynamics import PhaseSpacePosition
from gala.units import galactic

from astropy.constants import G
import matplotlib.pyplot as plt
from astropy.table import Table, Column

np.set_printoptions(suppress=True)
from matplotlib.legend_handler import HandlerPathCollection

from astropy.coordinates import Distance
from astropy.coordinates import Galactocentric
from astropy.units import Quantity
from astropy.coordinates import Galactocentric

import pandas as pd

from astroquery.gaia import Gaia
import fitsio
import time

In [ ]:
Gahla_raw_allstar = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/galah_dr4_allstar_240705.fits'
Gahla_raw_dynamics = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/galah_dr4_vac_dynamics_240705.fits'
Pradosh_fits_file = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/Prodosh_catolgue_full_data.fits'
Pradosh_fits_file_2 = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/Paper_2_total_catalogue_with_flags_errors_kinematics_MNRAS_ready_fits'


# ED_2_Stream = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/ED_2.fits'
Apogee_allstaraspcap_fits = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/astraAllStarASPCAP-0.6.0.fits'

ED_2_Stream = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/ED_2.csv'
Gahla_all_gaia_kinematics_csv_data = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/Gahla_all_gaia_kinematics.csv'
Pradosh_all_gaia_kinematics_csv_data = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/Pradosh_all_gaia_kinematics.csv'
Pradosh_all_gaia_kinematics_csv_data_2 = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/Jupyter Notebooks/Pradosh_all_gaia_kinematics.csv'
# Apogee_allstaraspcap_csv_file = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/fits and csv files/Apogee_Allstaraspcap.csv'
# Apogee_all_gaia_kinematics_csv_data = "/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/Jupyter Notebooks/Apogee_all_gaia_kinematics.csv"

def ensure_native_endian(df):
    """Convert all numeric NumPy columns in a DataFrame to native byte order."""
    for col in df.columns:
        dtype = df[col].dtype

        # Only handle real NumPy dtypes (skip pandas extension dtypes like StringDtype)
        if not isinstance(dtype, np.dtype):
            continue

        # Skip already-native or non-byte-order dtypes
        if dtype.byteorder in ('=', '|'):
            continue

        if np.issubdtype(dtype, np.number):
            df[col] = df[col].astype(dtype.newbyteorder('='))
    
    return df

In [ ]:
# ED_2_STREAM = Table.read(ED_2_Stream, format="csv")


Gahla_all_gaia_kinematics_csv = Table.read(Gahla_all_gaia_kinematics_csv_data, format="csv")
Gahla_all_gaia_kinematics_csv = Gahla_all_gaia_kinematics_csv[
    Gahla_all_gaia_kinematics_csv['parallax'] > 0
]


# Pradosh_all_gaia_kinematics_csv_data = Table.read(Pradosh_all_gaia_kinematics_csv_data, format="csv")
# Pradosh_all_gaia_kinematics_csv_data = Pradosh_all_gaia_kinematics_csv_data[
#     Pradosh_all_gaia_kinematics_csv_data['parallax'] > 0
# ]


# Apogee_allstaraspcap_csv = Table.read(Apogee_allstaraspcap_csv_file, format="csv")
# Apogee_allstaraspcap_csv = Apogee_allstaraspcap_csv[
#     Apogee_allstaraspcap_csv['parallax'] > 0
# ]

# Apogee_all_Gaia_kinematics_csv = Table.read(Apogee_all_gaia_kinematics_csv_data, format="csv")
# Apogee_all_Gaia_kinematics_csv = Apogee_all_Gaia_kinematics_csv[
#     Apogee_all_Gaia_kinematics_csv['parallax'] > 0
# ]


In [ ]:
with fitsio.FITS(Gahla_raw_allstar) as hdul1:
    data_1 = hdul1[1].read()
Gahla_Raw_Allstar = pd.DataFrame(data_1)

with fitsio.FITS(Gahla_raw_dynamics) as hdul1:
    data_1 = hdul1[1].read()
Gahla_Raw_Dynamics = pd.DataFrame(data_1)
Gahla_Raw_Dynamics = Gahla_Raw_Dynamics.drop(columns=['tmass_id', 'gaiadr3_source_id', 'r_med', 'dec', 'ra'])

Gahla_Raw = pd.merge(Gahla_Raw_Allstar, Gahla_Raw_Dynamics, on="sobject_id", how="inner")
list(Gahla_Raw.columns)

# with fitsio.FITS(ED_2_Stream) as hdul1:
#     data_1 = hdul1[1].read()
# ED_2_STREAM = pd.DataFrame(data_1)

# with fitsio.FITS(Pradosh_fits_file_2) as hdul1:
#     data_1 = hdul1[1].read()
# Pradosh_Fits_File = pd.DataFrame(data_1)

In [ ]:
def extract_6d_params(t):

    ra = t['ra']*u.deg
    pmra = t['pmra']*u.mas/u.yr
    dec = t['dec']*u.deg
    pmdec = t['pmdec']*u.mas/u.yr
    par = t['parallax']

    ##### NEW
    # dist = Distance(parallax=t['parallax']*u.mas)
    ##### NEW
    dist = 1/par
    dist = dist*u.kpc

    rv = t['radial_velocity']*u.km/u.s # Radial velocity in km/s

    c = SkyCoord(ra=ra,dec=dec,distance = dist, pm_ra_cosdec=pmra,pm_dec=pmdec,radial_velocity=rv, frame='icrs')

    return c

def convert_to_galactic(coordinates):
    galcen_frame = Galactocentric(galcen_distance=8.122*u.kpc, z_sun=20.8*u.pc, galcen_v_sun=[11.1, 12.24 + 232.0, 7.25] * u.km/u.s)
    c_galcen = coordinates.transform_to(galcen_frame)

    pos = c_galcen.cartesian.without_differentials()
    vel = c_galcen.velocity.d_xyz

    T = Table()
    T['X'] = pos.x
    T['Y'] = pos.y
    T['Z'] = pos.z
    T['U'] = vel[0]
    T['V'] = vel[1]
    T['W'] = vel[2]

    return T

def calculate_ELx(cart_T):
    """"
    Compute Lz, L_perp, and energy using galpy's MWPotential2014.

    Parameters:
    -----------
    cart_T: dict-like with keys:
        'X', 'Y', 'Z' in kpc
        'U', 'V', 'W' in km/s

    Returns:
    --------
    tuple: (Lz, L_perp, energy) in km*kpc/s and km^2/s^2
    """
    # Galpy scale constants
    ro = 8.122 * u.kpc
    vo = 232.0 * u.km / u.s

    # Get positions and velocities (already in kpc, km/s)
    # Ensure all inputs have correct units
    X = cart_T['X'] * u.kpc if not hasattr(cart_T['X'], 'unit') else cart_T['X'].to(u.kpc)
    Y = cart_T['Y'] * u.kpc if not hasattr(cart_T['Y'], 'unit') else cart_T['Y'].to(u.kpc)
    Z = cart_T['Z'] * u.kpc if not hasattr(cart_T['Z'], 'unit') else cart_T['Z'].to(u.kpc)

    U = cart_T['U'] * u.km/u.s if not hasattr(cart_T['U'], 'unit') else cart_T['U'].to(u.km/u.s)
    V = cart_T['V'] * u.km/u.s if not hasattr(cart_T['V'], 'unit') else cart_T['V'].to(u.km/u.s)
    W = cart_T['W'] * u.km/u.s if not hasattr(cart_T['W'], 'unit') else cart_T['W'].to(u.km/u.s)





    #Apply Solar motion corrections to velocities
    U_gal = U
    V_gal = V
    W_gal = W





    # Convert to galpy orbit inputs (scaled)
    R = np.sqrt(X**2 + Y**2)
    phi = np.arctan2(Y, X)

    # Cartesian velocities
    vX = U_gal
    vY = V_gal

    # Convert to cylindrical velocities
    vR = (X * vX + Y * vY) / R
    vT = (-X * vY + Y * vX) / R # Azimuthal velocity (v_phi)

    orb = Orbit(vxvv=[
        (R / ro).to_value(u.dimensionless_unscaled),
        (vR / vo).to_value(u.dimensionless_unscaled),
        (vT / vo).to_value(u.dimensionless_unscaled),
        (Z / ro).to_value(u.dimensionless_unscaled),
        (W_gal / vo).to_value(u.dimensionless_unscaled),
        phi.to_value(u.rad)], ro=ro.to_value(u.kpc), vo=vo.to_value(u.km/u.s))





    # Convert to physical units (km²/s²)
    #energy = E #* vo.to_value(u.km/u.s)**2  # convert to km²/s²

    # Angular momentum
    Lx = Y * W_gal - Z * V_gal
    Ly = Z * U_gal - X * W_gal





    Lx = Lx = Y * W_gal - Z * V_gal
    Ly = Z * U_gal - X * W_gal
    Lz = orb.Lz(pot=MWPotential2014)#.to(u.kpc * u.km/u.s)
    L_perp = np.hypot(Lx, Ly)
    energy = orb.E(pot=MWPotential2014) #* vo.to_value(u.km/u.s)**2  # convert to km²/s²
    jr = orb.jr(pot=MWPotential2014)
    jphi = orb.jp(pot=MWPotential2014)
    jz = orb.jz(pot=MWPotential2014)
    ecc = orb.e(pot=MWPotential2014, analytic=True)
    vr = orb.vr(pot=MWPotential2014)
    vphi = orb.vT(pot=MWPotential2014)
    vz = orb.vz(pot=MWPotential2014)
    vnet = np.sqrt(vr**2 + vz**2)
    #### NEW

    return Lx, Ly, Lz, L_perp, energy, jr, jphi, jz, ecc, vr, vphi, vz, vnet

In [ ]:
ED_2_6d_parametars = extract_6d_params(ED_2_STREAM)

ED_2_coordinates = convert_to_galactic(ED_2_6d_parametars)

ED_2_Lx, ED_2_Ly, ED_2_Lz, ED_2_L_perp, ED_2_energy, ED_2_jr, ED_2_jphi, ED_2_jz, ED_2_ecc, ED_2_vr, ED_2_vphi, ED_2_vz, ED_2_vnet = calculate_ELx(ED_2_coordinates)

In [ ]:
ED_2_kinematics = Table()

ED_2_kinematics['source_id'] = ED_2_STREAM['source_id']
ED_2_kinematics['Lx'] = ED_2_Lx
ED_2_kinematics['Ly'] = ED_2_Ly
ED_2_kinematics['Lz'] = ED_2_Lz
ED_2_kinematics['L_perp'] = ED_2_L_perp
ED_2_kinematics['energy'] = ED_2_energy
ED_2_kinematics['jr'] = ED_2_jr
ED_2_kinematics['jphi'] = ED_2_jphi
ED_2_kinematics['jz'] = ED_2_jz
ED_2_kinematics['jtot'] = (np.fabs(ED_2_jphi) + ED_2_jz + ED_2_jr)
ED_2_kinematics['ecc'] = ED_2_ecc
ED_2_kinematics['vr'] = ED_2_vr
ED_2_kinematics['vphi'] = ED_2_vphi
ED_2_kinematics['vz'] = ED_2_vz
ED_2_kinematics['vnet'] = ED_2_vnet

ED_2_kinematics

In [ ]:
ED_2_kinematics = ED_2_kinematics.to_pandas()
# ED_2_kinematics.to_csv('ED_2__gaia_kinematics.csv', sep=',', index=False)

In [ ]:
gahla_chunks = {}
Gahla_other_kinematics = []

total = len(Gahla_Raw['gaiadr3_source_id'])
step = 50000

source_ids = Gahla_Raw['gaiadr3_source_id']
start_time = time.time()

for start in range(0, total, step):
    end = min(start + step, total)
    print('start: ',start)
    print('end: ', end)
    print(f'elapsed time: {time.time() - start_time:.2f} s')
    
    key = f"source_ids_{start}_{end}"
    gahla_chunks[key] = source_ids.iloc[start:end]

print(gahla_chunks.keys())
for key in gahla_chunks.keys():
    # if key == 'source_ids_0_20000':

    current_time = time.time()
    print( '----------------------------------------------')
    print('')
    print(key, "     elapsed time: ", current_time - start_time)
    print('')

    source_ids = gahla_chunks[key]
    print("length of source_ids: ", len(source_ids))
    ids_str = ",".join(str(i) for i in source_ids)
    print('len of ids_str: ', len(ids_str))
    # print(ids_str)

    query = f"""
    SELECT 
    source_id, radial_velocity, ra, pmra, dec, pmdec, parallax
    FROM gaiadr3.gaia_source_lite
    WHERE source_id IN ({ids_str})
    AND radial_velocity is not NULL
    AND ra IS NOT NULL
    AND pmra IS NOT NULL
    AND dec IS NOT NULL
    AND pmdec IS NOT NULL
    AND parallax IS NOT NULL
    """
    #   source_id, radial_velocity, radial_velocity_error, ra, pmra,pmra_error, dec, pmdec, pmdec_error, parallax, parallax_error
    # AND radial_velocity is not NULL



    job = Gaia.launch_job_async(query)
    tbl = job.get_results()
    print(tbl)
    # Gahla_chunk_kinematics = tbl.to_pandas()
    Gahla_other_kinematics.append(tbl.to_pandas())
    # print(Gahla_other_kinematics)

    # Gahla_other_kinematics = ensure_native_endian(Gahla_other_kinematics)
    # print(Gahla_other_kinematics.head())
    # print(Gahla_other_kinematics.columns)
    # print(Gahla_other_kinematics.shape)

Gahla_all_gaia_kinematics = pd.concat(Gahla_other_kinematics, ignore_index=True)
# Gahla_all_gaia_kinematics.to_csv('Gahla_all_gaia_kinematics.csv', sep=',', index=False)

In [ ]:
Gahla_all_gaia_parametars = extract_6d_params(Gahla_all_gaia_kinematics_csv)

Gahla_all_gaia_coordinates = convert_to_galactic(Gahla_all_gaia_parametars)

Gahla_all_gaia_Lx, Gahla_all_gaia_Ly, Gahla_all_gaia_Lz, Gahla_all_gaia_L_perp, Gahla_all_gaia_energy, Gahla_all_gaia_jr, Gahla_all_gaia_jphi, Gahla_all_gaia_jz, Gahla_all_gaia_ecc, Gahla_all_gaia_vr, Gahla_all_gaia_vphi, Gahla_all_gaia_vz, Gahla_all_gaia_vnet = calculate_ELx(Gahla_all_gaia_coordinates)

In [ ]:
Gahla_all_GAIA_kinematics = Table()

Gahla_all_GAIA_kinematics['source_id'] = Gahla_all_gaia_kinematics_csv['source_id']
Gahla_all_GAIA_kinematics['radial_velocity'] = Gahla_all_gaia_kinematics_csv['radial_velocity']
Gahla_all_GAIA_kinematics['ra'] = Gahla_all_gaia_kinematics_csv['ra']
Gahla_all_GAIA_kinematics['dec'] = Gahla_all_gaia_kinematics_csv['dec']
Gahla_all_GAIA_kinematics['pmra'] = Gahla_all_gaia_kinematics_csv['pmra']
Gahla_all_GAIA_kinematics['pmdec'] = Gahla_all_gaia_kinematics_csv['pmdec']
Gahla_all_GAIA_kinematics['parallax'] = Gahla_all_gaia_kinematics_csv['parallax']

Gahla_all_GAIA_kinematics['Lx'] = Gahla_all_gaia_Lx
Gahla_all_GAIA_kinematics['Ly'] = Gahla_all_gaia_Ly
Gahla_all_GAIA_kinematics['Lz'] = Gahla_all_gaia_Lz
Gahla_all_GAIA_kinematics['L_perp'] = Gahla_all_gaia_L_perp
Gahla_all_GAIA_kinematics['energy'] = Gahla_all_gaia_energy
Gahla_all_GAIA_kinematics['jr'] = Gahla_all_gaia_jr
Gahla_all_GAIA_kinematics['jphi'] = Gahla_all_gaia_jphi
Gahla_all_GAIA_kinematics['jz'] = Gahla_all_gaia_jz
Gahla_all_GAIA_kinematics['jtot'] = (np.fabs(Gahla_all_gaia_jphi) + Gahla_all_gaia_jz + Gahla_all_gaia_jr)
Gahla_all_GAIA_kinematics['ecc'] = Gahla_all_gaia_ecc
Gahla_all_GAIA_kinematics['vr'] = Gahla_all_gaia_vr
Gahla_all_GAIA_kinematics['vphi'] = Gahla_all_gaia_vphi
Gahla_all_GAIA_kinematics['vz'] = Gahla_all_gaia_vz
Gahla_all_GAIA_kinematics['vnet'] = Gahla_all_gaia_vnet

Gahla_all_GAIA_kinematics = Gahla_all_GAIA_kinematics.to_pandas()

Gahla_gaia_merge = pd.merge(
    Gahla_Raw,
    Gahla_all_GAIA_kinematics,
    left_on='gaiadr3_source_id',
    right_on='source_id',
    how="outer",
    suffixes=("_gahla", "_gaia")
)

In [ ]:
for col in Gahla_gaia_merge.columns:
    print(col)

In [ ]:
Gahla_gaia_merge.to_csv(
    "Gahla_all_GAIA_kinematics_Aug_2_using_previous_data.csv",
    index=False
)

In [ ]:
Pradosh_chunks = {} # change name here
Pradosh_other_kinematics = [] # change name here

# total = len(Pradosh_Fits_File['source_id']) # change name here
total = len(Pradosh_Fits_File['Gaia_dr3_id']) # change name here
step = 2000

source_ids = Pradosh_Fits_File['Gaia_dr3_id'] # change name here
start_time = time.time()

for start in range(0, total, step):

    end = min(start + step, total)

    print('----------------------------------------------')
    print(f'start: {start}')
    print(f'end: {end}')
    print(f'elapsed time: {time.time() - start_time:.2f} s')

    key = f"source_ids_{start}_{end}"
    Pradosh_chunks[key] = source_ids.iloc[start:end]

print(Pradosh_chunks.keys()) # fix name here
for key in Pradosh_chunks.keys():

    current_time = time.time()
    print( '----------------------------------------------')
    print('')
    print(key, "     elapsed time: ", current_time - start_time)
    print('')

    source_ids = Pradosh_chunks[key]
    print("length of source_ids: ", len(source_ids))
    ids_str = ",".join(str(i) for i in source_ids)
    print('len of ids_str: ', len(ids_str))

    query = f"""
    SELECT 
    source_id, radial_velocity, ra, pmra, dec, pmdec, parallax
    FROM gaiadr3.gaia_source_lite
    WHERE source_id IN ({ids_str})
    AND radial_velocity is not NULL
    """
    #   source_id, radial_velocity, radial_velocity_error, ra, pmra,pmra_error, dec, pmdec, pmdec_error, parallax, parallax_error

    # AND ra IS NOT NULL
    # AND pmra IS NOT NULL
    # AND dec IS NOT NULL
    # AND pmdec IS NOT NULL
    # AND parallax IS NOT NULL

    job = Gaia.launch_job(query)
    tbl = job.get_results()
    print(tbl)
    # Gahla_chunk_kinematics = tbl.to_pandas()
    Pradosh_other_kinematics.append(tbl.to_pandas())
    # print(Gahla_other_kinematics)

    # Gahla_other_kinematics = ensure_native_endian(Gahla_other_kinematics)
    # print(Gahla_other_kinematics.head())
    # print(Gahla_other_kinematics.columns)
    # print(Gahla_other_kinematics.shape)

Pradosh_all_gaia_kinematics = pd.concat(Pradosh_other_kinematics, ignore_index=True)
# Pradosh_all_gaia_kinematics.to_csv('Pradosh_all_gaia_kinematics.csv', sep=',', index=False)

In [ ]:
# Pradosh_all_gaia_kinematics.to_csv('Pradosh_all_gaia_kinematics.csv', sep=',', index=False)

In [ ]:
Pradosh_all_gaia_kinematics_csv_data = Table.read(Pradosh_all_gaia_kinematics_csv_data_2, format="csv")
Pradosh_all_gaia_kinematics_csv_data = Pradosh_all_gaia_kinematics_csv_data[
    Pradosh_all_gaia_kinematics_csv_data['parallax'] > 0
]

In [ ]:
Pradosh_all_gaia_parametars = extract_6d_params(Pradosh_all_gaia_kinematics_csv_data)

Pradosh_all_gaia_coordinates = convert_to_galactic(Pradosh_all_gaia_parametars)

Pradosh_all_gaia_Lx, Pradosh_all_gaia_Ly, Pradosh_all_gaia_Lz, Pradosh_all_gaia_L_perp, Pradosh_all_gaia_energy, Pradosh_all_gaia_jr, Pradosh_all_gaia_jphi, Pradosh_all_gaia_jz, Pradosh_all_gaia_ecc, Pradosh_all_gaia_vr, Pradosh_all_gaia_vphi, Pradosh_all_gaia_vz, Pradosh_all_gaia_vnet = calculate_ELx(Pradosh_all_gaia_coordinates)

In [ ]:
Pradosh_all_GAIA_kinematics = Table()

Pradosh_all_GAIA_kinematics['source_id'] = Pradosh_all_gaia_kinematics_csv_data['source_id']
# Pradosh_all_GAIA_kinematics['Lx'] = Pradosh_all_gaia_Lx
# Pradosh_all_GAIA_kinematics['Ly'] = Pradosh_all_gaia_Ly
# Pradosh_all_GAIA_kinematics['Lz'] = Pradosh_all_gaia_Lz
# Pradosh_all_GAIA_kinematics['L_perp'] = Pradosh_all_gaia_L_perp
# Pradosh_all_GAIA_kinematics['energy'] = Pradosh_all_gaia_energy
Pradosh_all_GAIA_kinematics['jr'] = Pradosh_all_gaia_jr
Pradosh_all_GAIA_kinematics['jphi'] = Pradosh_all_gaia_jphi
Pradosh_all_GAIA_kinematics['jz'] = Pradosh_all_gaia_jz
Pradosh_all_GAIA_kinematics['jtot'] = (np.fabs(Pradosh_all_gaia_jphi) + Pradosh_all_gaia_jz + Pradosh_all_gaia_jr)
# Pradosh_all_GAIA_kinematics['ecc'] = Pradosh_all_gaia_ecc
# Pradosh_all_GAIA_kinematics['vr'] = Pradosh_all_gaia_vr
# Pradosh_all_GAIA_kinematics['vphi'] = Pradosh_all_gaia_vphi
# Pradosh_all_GAIA_kinematics['vz'] = Pradosh_all_gaia_vz
# Pradosh_all_GAIA_kinematics['vnet'] = Pradosh_all_gaia_vnet

Pradosh_all_GAIA_kinematics = Pradosh_all_GAIA_kinematics.to_pandas()

Pradosh_gaia_merge = pd.merge(
    Pradosh_Fits_File,
    Pradosh_all_GAIA_kinematics,
    left_on='Gaia_dr3_id',
    right_on='source_id',
    how="inner",
    # suffixes=("_Pradosh", "_gaia")
)

In [ ]:
Pradosh_gaia_merge = Pradosh_gaia_merge.drop(columns=["ra", "dec", 'parallax'])

In [ ]:
# Pradosh_gaia_merge.to_csv(
#     "Pradosh_revision_kinematics.csv",
#     index=False
# )

# Pradosh_gaia_merge.to_fits = Table.from_pandas(Pradosh_gaia_merge)
# Pradosh_gaia_merge.to_fits.write("Pradosh_revision_kinematics.fits", overwrite=True)

In [ ]:
#this cell is to convert a fits file into a csv

# with fitsio.FITS(Apogee_allstaraspcap_fits) as f:
#     print(f)

with fitsio.FITS(Apogee_allstaraspcap_fits) as hdul1:
    data_1 = hdul1[2].read()
    one_d_cols = [
        name for name in data_1.dtype.names
        if data_1[name].ndim == 1
    ]
    # Build a smaller structured array
    data_1 = data_1[one_d_cols]
Apogee_Allstaraspcap_fits = pd.DataFrame(data_1)


In [ ]:
Apogee_Allstaraspcap_fits_flagged = Apogee_Allstaraspcap_fits[(Apogee_Allstaraspcap_fits['result_flags'] == 0)]# & Apogee_Allstaraspcap_fits['flag_bad'] == False]


''' what is wrtieen under hre is good for the MWpotenial code to work, but I dont know if it usng the same mesurments exactly. 
    So if I used Gaia data for Galah o=and Pradosh it wouldn't be correct to not use it as well for Apogee
'''
Apogee_Allstaraspcap_fits_flagged['pmdec'] = Apogee_Allstaraspcap_fits_flagged['pmde']   
# Apogee_Allstaraspcap_fits_flagged['radial_velocity'] = Apogee_Allstaraspcap_fits_flagged['gaia_v_rad']
Apogee_Allstaraspcap_fits_flagged['radial_velocity'] = Apogee_Allstaraspcap_fits_flagged['v_rad']
Apogee_Allstaraspcap_fits_flagged['parallax'] = Apogee_Allstaraspcap_fits_flagged['plx']
# Apogee_Allstaraspcap_fits_flagged = Apogee_Allstaraspcap_fits_flagged[Apogee_Allstaraspcap_fits_flagged['']]
# # Apogee_Allstaraspcap_fits_flagged.to_csv('Apogee_Allstaraspcap.csv', sep=',', index=False)

Apogee_Allstaraspcap_fits_flagged = Apogee_Allstaraspcap_fits_flagged[(Apogee_Allstaraspcap_fits_flagged['ra'].notna())
                                                                      & (Apogee_Allstaraspcap_fits_flagged['pmra'].notna())
                                                                      & (Apogee_Allstaraspcap_fits_flagged['dec'].notna())
                                                                      & (Apogee_Allstaraspcap_fits_flagged['pmdec'].notna())
                                                                      & (Apogee_Allstaraspcap_fits_flagged['radial_velocity'].notna())
                                                                      & (Apogee_Allstaraspcap_fits_flagged['parallax'].notna())
                                                                      & (Apogee_Allstaraspcap_fits_flagged['parallax'] > 0)
                                                                      ]

# Apogee_Kinematics_inputs = Apogee_Allstaraspcap_fits_flagged.to_csv(sep=',', index=False)

Apogee_Kinematics_inputs = Table()

Apogee_Kinematics_inputs['ra'] = Apogee_Allstaraspcap_fits_flagged['ra']
Apogee_Kinematics_inputs['dec'] = Apogee_Allstaraspcap_fits_flagged['dec']
Apogee_Kinematics_inputs['pmra'] = Apogee_Allstaraspcap_fits_flagged['pmra']
Apogee_Kinematics_inputs['pmdec'] = Apogee_Allstaraspcap_fits_flagged['pmdec']
Apogee_Kinematics_inputs['radial_velocity'] = Apogee_Allstaraspcap_fits_flagged['radial_velocity']
Apogee_Kinematics_inputs['parallax'] = Apogee_Allstaraspcap_fits_flagged['parallax']


In [ ]:
Apogee_Kinematics_inputs

In [ ]:
# Apogee_Allstaraspcap_fits_flagged.to_csv('Apogee_Allstaraspcap.csv', sep=',', index=False)

In [ ]:
Apogee_chunks = {}
Apogee_other_kinematics = []

total = len(Apogee_Allstaraspcap_fits_flagged['gaia_dr3_source_id'])
step = 50000

source_ids = Apogee_Allstaraspcap_fits_flagged['gaia_dr3_source_id']
for start in range(0, total, step):
    end = min(start + step, total)
    print('start: ',start)
    print('end: ', end)
    key = f"source_ids_{start}_{end}"
    Apogee_chunks[key] = source_ids.iloc[start:end]

print(Apogee_chunks.keys())
start_time = time.time()
for key in Apogee_chunks.keys():
    # if key == 'source_ids_0_20000':

    current_time = time.time()
    print( '----------------------------------------------')
    print('')
    print(key, "     elapsed time: ", current_time - start_time)
    print('')

    source_ids = Apogee_chunks[key]
    print("length of source_ids: ", len(source_ids))
    ids_str = ",".join(str(i) for i in source_ids)
    print('len of ids_str: ', len(ids_str))
    # print(ids_str)

    query = f"""
    SELECT 
    source_id, radial_velocity, ra, pmra, dec, pmdec, parallax
    FROM gaiadr3.gaia_source_lite
    WHERE source_id IN ({ids_str})
    AND radial_velocity is not NULL
    AND ra IS NOT NULL
    AND pmra IS NOT NULL
    AND dec IS NOT NULL
    AND pmdec IS NOT NULL
    AND parallax IS NOT NULL
    """
    #   source_id, radial_velocity, radial_velocity_error, ra, pmra,pmra_error, dec, pmdec, pmdec_error, parallax, parallax_error
    # AND radial_velocity is not NULL



    job = Gaia.launch_job_async(query)
    tbl = job.get_results()
    print(tbl)
    # Apogee_other_kinematics = tbl.to_pandas()
    Apogee_other_kinematics.append(tbl.to_pandas())
    # print(Apogee_other_kinematics)

    # Apogee_other_kinematics = ensure_native_endian(Apogee_other_kinematics)
    # print(Apogee_other_kinematics.head())
    # print(Apogee_other_kinematics.columns)
    # print(Apogee_other_kinematics.shape)

Apogee_all_gaia_kinematics = pd.concat(Apogee_other_kinematics, ignore_index=True)
# Apogee_all_gaia_kinematics.to_csv('Apogee_all_gaia_kinematics.csv', sep=',', index=False)

In [ ]:
# Apogee_all_gaia_kinematics.to_csv('Apogee_all_gaia_kinematics.csv', sep=',', index=False)

In [ ]:
# Apogee_Allstaraspcap_fits
# Apogee_Allstaraspcap_fits_flagged
# Apogee_Allstaraspcap_fits_flagged_bad
# Apogee_Allstaraspcap_fits['flag_bad']
# Apogee_Allstaraspcap_fits_flagged['flag_bad']

# Apogee_Allstaraspcap_fits
# Apogee_Allstaraspcap_fits_flagged
# Apogee_Allstaraspcap_fits_flagged_bad
# Apogee_Allstaraspcap_fits['flag_bad']
# Apogee_Allstaraspcap_fits_flagged['flag_bad']

In [ ]:
# Apogee_parameters = extract_6d_params(Apogee_all_Gaia_kinematics_csv) # this one used the values given by Gaia baced on the gaia dr3 ID
Apogee_parameters = extract_6d_params(Apogee_Kinematics_inputs) # this used the values used/calculated by Apogee

Apogee_coordinates = convert_to_galactic(Apogee_parameters)

Apogee_all_gaia_Lx, Apogee_all_gaia_Ly, Apogee_all_gaia_Lz, Apogee_all_gaia_L_perp, Apogee_all_gaia_energy, Apogee_all_gaia_jr, Apogee_all_gaia_jphi, Apogee_all_gaia_jz, Apogee_all_gaia_ecc, Apogee_all_gaia_vr, Apogee_all_gaia_vphi, Apogee_all_gaia_vz, Apogee_all_gaia_vnet = calculate_ELx(Apogee_coordinates)

In [ ]:
# Apogee_allstaraspcap_pandas = Apogee_allstaraspcap_csv.to_pandas()

Apogee_all_Gaia_kinematics_pandas = Apogee_all_Gaia_kinematics_csv.to_pandas()

In [ ]:
Apogee_with_kinematics = Table()

Apogee_with_kinematics['gaia_dr3_source_id'] = Apogee_Allstaraspcap_fits_flagged['gaia_dr3_source_id']
# Apogee_with_kinematics['gaia_dr3_source_id'] = Apogee_all_Gaia_kinematics_pandas['source_id']
# Apogee_with_kinematics['ra'] = Apogee_all_Gaia_kinematics_pandas['ra']
# Apogee_with_kinematics['dec'] = Apogee_all_Gaia_kinematics_pandas['dec']
# Apogee_with_kinematics['pmra'] = Apogee_all_Gaia_kinematics_pandas['pmra']
# Apogee_with_kinematics['pmde'] = Apogee_all_Gaia_kinematics_pandas['pmdec']
# Apogee_with_kinematics['radial_velocity'] = Apogee_all_Gaia_kinematics_pandas['radial_velocity']
Apogee_with_kinematics['Lx'] = Apogee_all_gaia_Lx
Apogee_with_kinematics['Ly'] = Apogee_all_gaia_Ly
Apogee_with_kinematics['Lz'] = Apogee_all_gaia_Lz
Apogee_with_kinematics['L_perp'] = Apogee_all_gaia_L_perp
Apogee_with_kinematics['energy'] = Apogee_all_gaia_energy
Apogee_with_kinematics['jr'] = Apogee_all_gaia_jr
Apogee_with_kinematics['jphi'] = Apogee_all_gaia_jphi
Apogee_with_kinematics['jz'] = Apogee_all_gaia_jz
Apogee_with_kinematics['jtot'] = (np.fabs(Apogee_all_gaia_jphi) + Apogee_all_gaia_jz + Apogee_all_gaia_jr)
Apogee_with_kinematics['ecc'] = Apogee_all_gaia_ecc
Apogee_with_kinematics['vr'] = Apogee_all_gaia_vr
Apogee_with_kinematics['vphi'] = Apogee_all_gaia_vphi
Apogee_with_kinematics['vz'] = Apogee_all_gaia_vz
Apogee_with_kinematics['vnet'] = Apogee_all_gaia_vnet


Apogee_with_kinematics = Apogee_with_kinematics.to_pandas()

Apogee_with_kinematics_merge = pd.merge(
    # Apogee_Allstaraspcap_fits,
    Apogee_Allstaraspcap_fits_flagged,
    Apogee_with_kinematics,
    on='gaia_dr3_source_id',
    how="inner",
    suffixes=("_Apogee", "_gaia")
)

In [ ]:
Apogee_with_kinematics_merge

In [ ]:
for col in Apogee_with_kinematics_merge.columns:
    print(col)

In [ ]:
# Apogee_with_kinematics_merge.to_csv(
#     "Apogee_with_kinematics.csv",
#     index=False
# )

table = Table.from_pandas(Apogee_with_kinematics_merge)
table.write('Apogee_with_kinematics_from_apo_values.fits', format='fits', overwrite=True)